# Group 2 CPECOG2: Coral bleaching, Run 2 (Nov. 20)

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all (about 25 to 30 min).

This notebook continues from Run 1 and runs the tests in the Nov. 13 next steps:

1. **Data fix:** remove the 9 duplicate groups whose copies carry conflicting labels, and check the splits for near-duplicate leakage. The Run 1 split is otherwise kept.
2. **Macro F1 tracking:** a callback logs validation macro F1 and bleached recall every epoch and keeps the best-macro-F1 checkpoint.
3. **Baselines:** the majority-class check and the scratch CNN (3 convolutional blocks) on the same split.
4. **Tuning sweep (YOLOv8s-cls):** one setting changed at a time from the Run 1 settings, then the settings that helped are combined.
5. **Comparison model:** EfficientNet-B0 with ImageNet weights on the same split.
6. **Final configuration:** chosen on validation macro F1 only, retrained with 3 seeds, then evaluated **once** on the held-out test set and on the external BU + BHD set.
7. Model size, CPU latency, the list of test errors, and Grad-CAM heatmaps.

Outputs go to `results/run2/` and are zipped as `run2_results.zip`. The last cell prints a summary for the progress log.

In [ ]:
!pip -q install ultralytics==8.4.211 imagehash kagglehub huggingface_hub

In [ ]:
import os, glob, shutil, time, json, random, zipfile, copy
import numpy as np, pandas as pd
from PIL import Image
import imagehash, torch, torchvision
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
EPOCHS = 30                  # max epochs per run; early stopping with patience 8 (hyperparameter register)
PATIENCE = 8
IMGSZ, BATCH = 224, 32
CNN_EPOCHS = 40              # scratch CNN baseline
FINAL_SEEDS = [42, 43, 44]   # the final configuration is trained with these seeds (mean and std)
NEAR_DUP_T = 5               # pHash Hamming distance at or below which two images count as near-duplicates
EFFNET_WEIGHTS = "IMAGENET1K_V1"
DATA_SRC = None              # set a folder path here to skip the Kaggle download
EXT_SRC = None               # set a folder or .zip path here to skip the Hugging Face download
SPLITS_URL = "https://raw.githubusercontent.com/areslou/cpecog2-coral-bleaching/main/data/splits.csv"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
TDEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU:", GPU_NAME if torch.cuda.is_available() else "none. Runtime > Change runtime type > T4 GPU, or this will take hours.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
ROOT = os.path.join(BASE, "coral_data")              # padded-square images in the Ultralytics folder layout
OUT = os.path.join(BASE, "results", "run2"); os.makedirs(OUT, exist_ok=True)
CLASSES = ["healthy", "bleached"]                    # index 1 = bleached, the positive class
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
T_START = time.time()

## 1. Data: Run 1 split minus the conflicting-label duplicates, plus a near-duplicate leakage check

Run 1 kept the first copy of 9 photos that appear in both the healthy and the bleached folder. Their true label cannot be confirmed from the folders, so every copy is removed. All other images keep their Run 1 split (`data/splits.csv` in the repo), so Run 1 and Run 2 stay comparable and the test set has still never been evaluated.

Run 1 removed only exact perceptual-hash matches. Near-duplicates (pHash distance of 5 or less) that land in different splits leak information, so the validation or test copy is removed and the training copy is kept. Near-duplicate pairs that carry different labels are removed entirely, like the exact ones.

In [ ]:
if DATA_SRC is None:
    import kagglehub
    DATA_SRC = kagglehub.dataset_download("vencerlanz09/healthy-and-bleached-corals-image-classification")
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def label_of(p):
    parent = os.path.basename(os.path.dirname(p)).lower()
    if "bleach" in parent: return "bleached"
    if "health" in parent: return "healthy"
    return None

def phash_of(p):
    try:
        with Image.open(p) as im:
            return str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None

paths = sorted(p for p in glob.glob(os.path.join(DATA_SRC, "**", "*"), recursive=True) if p.lower().endswith(exts))
df = pd.DataFrame({"path": paths})
df["label"] = df["path"].map(label_of)
df["file"] = df["path"].map(os.path.basename)
df["phash"] = df["path"].map(phash_of)
df = df.dropna(subset=["label", "phash"]).reset_index(drop=True)
N_RAW = len(df)
print(N_RAW, "readable labeled images:", df["label"].value_counts().to_dict())

In [ ]:
splits1 = pd.read_csv(SPLITS_URL).rename(columns={"path": "file"})
df = df.merge(splits1[["file", "label", "split"]], on=["file", "label"], how="left")
SAME_SPLIT = (int(df["split"].notna().sum()) == len(splits1)) and not df.duplicated(["file", "label"]).any()
print(f"Run 1 split matched: {SAME_SPLIT} ({int(df['split'].notna().sum())} of {len(splits1)} rows)")
if not SAME_SPLIT:
    print("WARNING: splits.csv did not match the download, so the split is rebuilt the Run 1 way (it may differ from Run 1).")
    from sklearn.model_selection import train_test_split
    keep = df.drop_duplicates("phash", keep="first")
    tr, tmp = train_test_split(keep, test_size=0.30, stratify=keep["label"], random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=SEED)
    df["split"] = None
    df.loc[tr.index, "split"] = "train"; df.loc[va.index, "split"] = "val"; df.loc[te.index, "split"] = "test"

n_lab = df.groupby("phash")["label"].nunique()
conflict_hashes = sorted(n_lab[n_lab > 1].index)
conf = df[df["phash"].isin(conflict_hashes)].sort_values(["phash", "label"])
removed_conflict = conf[conf["split"].notna()]
print("Duplicate groups with conflicting labels:", len(conflict_hashes), f"({len(conf)} files)")
print("Copies removed from the Run 1 split:", removed_conflict["split"].value_counts().to_dict())

if len(conflict_hashes):
    groups = [g for _, g in conf.groupby("phash", sort=False)]
    per_row = 3; ncol = 2 * per_row; nrow = int(np.ceil(len(groups) / per_row))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.2 * ncol, 2.4 * nrow), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for g_i, g in enumerate(groups):
        for c_i, (_, r) in enumerate(g.head(2).iterrows()):
            ax = axes[g_i // per_row, (g_i % per_row) * 2 + c_i]
            ax.imshow(Image.open(r["path"]).convert("RGB")); ax.set_title(f"group {g_i + 1}: {r['label']}", fontsize=8)
    fig.suptitle("Same photo filed as both healthy and bleached (all copies removed)", fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "conflicting_duplicates.png"), dpi=150); plt.show()

In [ ]:
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def hamming(a, b):
    """Pairwise Hamming distances between two lists of 64-bit hex pHashes."""
    x = np.array([int(h, 16) for h in a], dtype=np.uint64)[:, None] ^ np.array([int(h, 16) for h in b], dtype=np.uint64)[None, :]
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int16)
    return POP[np.ascontiguousarray(x).view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1).astype(np.int16)

clean = df[df["split"].notna() & ~df["phash"].isin(conflict_hashes)].reset_index(drop=True)
D = hamming(clean["phash"], clean["phash"])
iu = np.triu_indices(len(clean), k=1)
print("Image pairs by pHash distance:", {k: int((D[iu] == k).sum()) for k in range(0, 11)})
sp, lb = clean["split"].to_numpy(), clean["label"].to_numpy()
ii, jj = np.where(np.triu(D <= NEAR_DUP_T, k=1))
pairs = list(zip(ii, jj))
label_conflict = [(i, j) for i, j in pairs if lb[i] != lb[j]]
cross = [(i, j) for i, j in pairs if sp[i] != sp[j] and lb[i] == lb[j]]
rank = {"train": 0, "val": 1, "test": 2}
leak_drop = {max((i, j), key=lambda k: rank[sp[k]]) for i, j in cross}       # keep the training copy
conflict_drop = {k for pair in label_conflict for k in pair}                 # label cannot be trusted
near_drop = sorted(leak_drop | conflict_drop)
print(f"Near-duplicate pairs (distance <= {NEAR_DUP_T}): {len(pairs)}; {len(cross)} same-label pairs across splits, "
      f"{len(label_conflict)} pairs with different labels")
print("Removed:", clean.loc[near_drop, "split"].value_counts().to_dict())

show = sorted(cross + label_conflict, key=lambda t: D[t])[:8]
if show:
    fig, axes = plt.subplots(len(show), 2, figsize=(4.6, 2.3 * len(show)), squeeze=False)
    for r_i, (i, j) in enumerate(show):
        for c_i, k in enumerate((i, j)):
            ax = axes[r_i, c_i]; ax.imshow(Image.open(clean.loc[k, "path"]).convert("RGB")); ax.axis("off")
            ax.set_title(f"{clean.loc[k, 'split']}, {clean.loc[k, 'label']} (d={D[i, j]})", fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "near_duplicates.png"), dpi=150); plt.show()

N_NEAR_PAIRS, N_CROSS, N_NEAR_CONFLICT = len(pairs), len(cross), len(label_conflict)
NEAR_REMOVED = clean.loc[near_drop, "split"].value_counts().to_dict()
clean = clean.drop(index=near_drop).reset_index(drop=True)
print(pd.crosstab(clean["split"], clean["label"], margins=True))
clean[["file", "label", "split", "phash"]].rename(columns={"file": "path"}).to_csv(os.path.join(OUT, "splits_v2.csv"), index=False)

In [ ]:
def pad_square(p):
    im = Image.open(p).convert("RGB"); s = max(im.size)
    sq = Image.new("RGB", (s, s), (0, 0, 0)); sq.paste(im, ((s - im.size[0]) // 2, (s - im.size[1]) // 2))
    return sq

shutil.rmtree(ROOT, ignore_errors=True)
for _, r in clean.iterrows():
    d = os.path.join(ROOT, r["split"], r["label"]); os.makedirs(d, exist_ok=True)
    pad_square(r["path"]).save(os.path.join(d, os.path.splitext(r["file"])[0] + ".jpg"), quality=95)

def split_files(split, root=ROOT):
    files = sorted(glob.glob(os.path.join(root, split, "*", "*.jpg")))
    return files, np.array([CLASSES.index(os.path.basename(os.path.dirname(p))) for p in files])

(tr_files, y_tr), (va_files, y_va), (te_files, y_te) = (split_files(s) for s in ("train", "val", "test"))
assert len(tr_files) + len(va_files) + len(te_files) == len(clean), "some images were not written (file name clash)"
COUNTS = {s: {"n": len(y), "bleached": int(y.sum()), "healthy": int((y == 0).sum())}
          for s, y in (("train", y_tr), ("val", y_va), ("test", y_te))}
print("Data written to", ROOT, "| train/val/test:", len(tr_files), len(va_files), len(te_files))

## 2. Metrics and the macro F1 callback

Ultralytics early stopping and `best.pt` use top-1 accuracy. The callback below reads the validator's predictions after every epoch, logs macro F1 and bleached recall, and copies the checkpoint to `best_f1.pt` whenever macro F1 improves. All model selection below uses `best_f1.pt`.

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score, confusion_matrix, classification_report
from ultralytics import YOLO
from ultralytics.utils.torch_utils import strip_optimizer

def metrics_from(y_true, p_bleached):
    """Metrics from P(bleached). The predicted class is bleached when P >= 0.5 (same as top-1 for two classes)."""
    p = np.asarray(p_bleached, dtype=float); y_pred = (p >= 0.5).astype(int)
    m = dict(macro_f1=f1_score(y_true, y_pred, average="macro"),
             bleached_recall=recall_score(y_true, y_pred, pos_label=1, zero_division=0),
             bleached_precision=precision_score(y_true, y_pred, pos_label=1, zero_division=0),
             healthy_recall=recall_score(y_true, y_pred, pos_label=0, zero_division=0),
             accuracy=accuracy_score(y_true, y_pred),
             roc_auc=roc_auc_score(y_true, p) if len(np.unique(p)) > 1 else 0.5)
    return m, y_pred

def track_macro_f1(model):
    """Log validation macro F1 and bleached recall each epoch; copy the checkpoint to best_f1.pt when macro F1 improves."""
    hist = []
    def on_fit_epoch_end(trainer):
        v = trainer.validator
        if v is None or not getattr(v, "training", False) or not v.pred:
            return  # skips the final re-validation of best.pt
        names = trainer.data["names"]
        names = names if isinstance(names, dict) else dict(enumerate(names))
        b = next(k for k, n in names.items() if n == "bleached")
        yp = (torch.cat(v.pred)[:, 0].numpy() == b).astype(int)
        yt = (torch.cat(v.targets).numpy() == b).astype(int)
        f1 = f1_score(yt, yp, average="macro")
        hist.append(dict(epoch=trainer.epoch + 1, macro_f1=f1, bleached_recall=recall_score(yt, yp, zero_division=0),
                         accuracy=float((yt == yp).mean())))
        if f1 > max((h["macro_f1"] for h in hist[:-1]), default=-1.0) and trainer.last.exists():
            shutil.copy(trainer.last, trainer.wdir / "best_f1.pt")
    model.add_callback("on_fit_epoch_end", on_fit_epoch_end)
    return hist

def yolo_run(name, stages, seed=SEED):
    """Train one YOLOv8s-cls configuration. Each stage is a dict of Ultralytics overrides; stage 2 starts from stage 1's best."""
    weights, rows, best, t0 = "yolov8s-cls.pt", [], None, time.time()
    for k, over in enumerate(stages):
        args = dict(data=ROOT, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=seed, deterministic=True, patience=PATIENCE,
                    project=os.path.join(BASE, "runs2"), name=f"{name}_s{seed}" + (f"_st{k + 1}" if len(stages) > 1 else ""),
                    exist_ok=True, device=DEVICE, plots=False, verbose=False)
        args.update(over)
        model = YOLO(weights)
        hist = track_macro_f1(model)
        model.train(**args)
        weights = str(model.trainer.wdir / "best_f1.pt")
        for h in hist:
            h["stage"] = k + 1
            if best is None or h["macro_f1"] > best["macro_f1"]:
                best = dict(h, weights=weights)
        rows += hist
    strip_optimizer(best["weights"])
    h = pd.DataFrame(rows); h["step"] = np.arange(1, len(h) + 1)
    settings = " | ".join(", ".join(f"{a}={b}" for a, b in s.items()) or "Run 1 settings" for s in stages)
    return dict(name=name, seed=seed, stages=stages, settings=settings, best_epoch=int(best["epoch"]), best_stage=int(best["stage"]),
                macro_f1=best["macro_f1"], bleached_recall=best["bleached_recall"], accuracy=best["accuracy"],
                epochs_run=len(h), minutes=(time.time() - t0) / 60, weights=best["weights"], hist=h)

def predict_yolo(weights, files, bs=64):
    m = YOLO(weights)
    b = next(k for k, n in m.names.items() if n == "bleached")
    p = []
    for k in range(0, len(files), bs):
        p += [float(r.probs.data[b]) for r in m.predict(files[k:k + bs], imgsz=IMGSZ, device=DEVICE, verbose=False)]
    return np.array(p)

## 3. Baselines: majority class and the scratch CNN

The scratch CNN has 3 convolutional blocks (two 3x3 convolutions with batch norm and ReLU, then max-pooling), global average pooling, dropout 0.3, and a linear layer. It is trained from random weights on the same split, with inputs normalized by the training-split mean and standard deviation, random flips and crops, AdamW (learning rate 1e-3, cosine decay), and the epoch with the best validation macro F1 kept.

In [ ]:
maj = int(np.bincount(y_tr).argmax())
maj_val, _ = metrics_from(y_va, np.full(len(y_va), float(maj)))
print(f"Majority-class check (always '{CLASSES[maj]}'): val macro F1 {maj_val['macro_f1']:.3f}")

def load_tensor(files, size=IMGSZ):
    """Images as a uint8 tensor (N, 3, size, size); the images are already padded to square."""
    return torch.stack([torch.from_numpy(np.array(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))).permute(2, 0, 1)
                        for p in files])

X_tr, X_va, X_te = load_tensor(tr_files), load_tensor(va_files), load_tensor(te_files)
MEAN = (X_tr.float() / 255).mean((0, 2, 3)); STD = (X_tr.float() / 255).std((0, 2, 3))   # training split only
print("Training-split mean / std:", MEAN.numpy().round(3), STD.numpy().round(3))

def augment(xb):
    """Random horizontal and vertical flips and a random crop of 80 to 100% of the side, resized back."""
    if random.random() < 0.5: xb = xb.flip(3)
    if random.random() < 0.5: xb = xb.flip(2)
    n = xb.shape[-1]; c = int(n * random.uniform(0.8, 1.0)); i, j = random.randint(0, n - c), random.randint(0, n - c)
    return F.interpolate(xb[..., i:i + c, j:j + c], size=(n, n), mode="bilinear", align_corners=False)

def predict_torch(net, X, mean, std, bs=64):
    net.eval(); out = []
    m_, s_ = mean.to(TDEV)[:, None, None], std.to(TDEV)[:, None, None]
    with torch.no_grad():
        for k in range(0, len(X), bs):
            out.append(net((X[k:k + bs].to(TDEV).float() / 255 - m_) / s_).softmax(1)[:, 1].cpu())
    return torch.cat(out).numpy()

def train_torch(net, mean, std, epochs, lr, wd=1e-4, seed=SEED):
    """Train a PyTorch classifier on the in-memory training split; keep the epoch with the best validation macro F1."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    net = net.to(TDEV)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    yt = torch.as_tensor(y_tr)
    m_, s_ = mean.to(TDEV)[:, None, None], std.to(TDEV)[:, None, None]
    best, hist = (-1.0, None, 0), []
    for ep in range(epochs):
        net.train(); perm = torch.randperm(len(X_tr))
        for k in range(0, len(perm), BATCH):
            idx = perm[k:k + BATCH]
            if len(idx) < 2: continue
            xb = augment((X_tr[idx].to(TDEV).float() / 255 - m_) / s_)
            loss = F.cross_entropy(net(xb), yt[idx].to(TDEV))
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        mv, _ = metrics_from(y_va, predict_torch(net, X_va, mean, std))
        hist.append(dict(epoch=ep + 1, **mv))
        if mv["macro_f1"] > best[0]:
            best = (mv["macro_f1"], copy.deepcopy(net.state_dict()), ep + 1)
    net.load_state_dict(best[1])
    return net, pd.DataFrame(hist), best[2]

class SmallCNN(nn.Module):
    """Baseline: 3 convolutional blocks trained from scratch, global average pooling, dropout, linear layer."""
    def __init__(self, n_classes=2):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_classes))
    def forward(self, x):
        return self.head(self.features(x))

t0 = time.time()
cnn, cnn_hist, cnn_ep = train_torch(SmallCNN(), MEAN, STD, epochs=CNN_EPOCHS, lr=1e-3)
cnn_min = (time.time() - t0) / 60
cnn_val, _ = metrics_from(y_va, predict_torch(cnn, X_va, MEAN, STD))
cnn_params = sum(p.numel() for p in cnn.parameters())
CNN_PATH = os.path.join(OUT, "baseline_cnn.pt")
torch.save({"state_dict": cnn.state_dict(), "mean": MEAN, "std": STD}, CNN_PATH)
cnn_hist.to_csv(os.path.join(OUT, "baseline_cnn_history.csv"), index=False)
print(f"Scratch CNN ({cnn_params / 1e6:.2f}M parameters, {cnn_min:.1f} min): val macro F1 {cnn_val['macro_f1']:.3f}, "
      f"bleached recall {cnn_val['bleached_recall']:.3f}, best epoch {cnn_ep} of {CNN_EPOCHS}")

## 4. Tuning sweep: YOLOv8s-cls, one setting at a time

Free Colab time rules out a full grid, so each run changes one group of settings from the reference (Run 1 settings on the cleaned split, up to 30 epochs, patience 8, best-macro-F1 checkpoint). All runs use seed 42 and are compared on validation macro F1 only.

| Run | Change from the reference | Why (hyperparameter register) |
| :-- | :-- | :-- |
| R2 | none (reference) | Run 1 settings on the cleaned data |
| R3 | light augmentation: RandAugment and random erasing off, flips, crop scale 0.1, small color shifts | strong color augmentation may hide the whitening cue |
| R4 | dropout 0.3 on the classification head | overfitting control |
| R5 | AdamW, lr0 5e-4, cosine decay | lower learning rate keeps more of the pretrained features |
| R6 | two stages: head only (backbone frozen, lr0 1e-3), then the top 2 backbone layers unfrozen (lr0 1e-4) | the training plan in the proposal |
| R7 | the settings from R3 to R6 that beat R2, combined | only runs if two or more settings helped |

In [ ]:
LIGHT_AUG = dict(auto_augment=None, erasing=0.0, scale=0.1, fliplr=0.5, flipud=0.5, hsv_h=0.0, hsv_s=0.2, hsv_v=0.1)
E1 = max(1, EPOCHS // 3)
SWEEP = {
    "R2_reference": [dict()],
    "R3_light_aug": [dict(LIGHT_AUG)],
    "R4_dropout0.3": [dict(dropout=0.3)],
    "R5_lr5e-4_cos": [dict(optimizer="AdamW", lr0=5e-4, cos_lr=True)],
    "R6_two_stage": [dict(freeze=9, epochs=E1, optimizer="AdamW", lr0=1e-3),
                     dict(freeze=7, epochs=EPOCHS - E1, optimizer="AdamW", lr0=1e-4, warmup_epochs=0)],
}
results = []
for name, stages in SWEEP.items():
    r = yolo_run(name, stages); results.append(r)
    print(f"\n>>> {name}: val macro F1 {r['macro_f1']:.3f}, bleached recall {r['bleached_recall']:.3f} "
          f"(epoch {r['best_epoch']} of stage {r['best_stage']}, {r['epochs_run']} epochs run, {r['minutes']:.1f} min)\n")

In [ ]:
ref = results[0]["macro_f1"]
better = [r["name"] for r in results[1:] if r["macro_f1"] > ref]
print("Settings that beat the reference on validation macro F1:", better or "none")
simple = {}
for n in better:
    if n != "R6_two_stage":
        simple.update(SWEEP[n][0])
if "R6_two_stage" in better:   # keep the stage learning rates; add the other settings that helped to both stages
    extra = {k: v for k, v in simple.items() if k not in ("optimizer", "lr0", "cos_lr")}
    combo = [{**SWEEP["R6_two_stage"][0], **extra}, {**SWEEP["R6_two_stage"][1], **extra}]
else:
    combo = [simple]
if len(better) >= 2:
    SWEEP["R7_combined"] = combo
    r = yolo_run("R7_combined", combo); results.append(r)
    print(f"\n>>> R7_combined: val macro F1 {r['macro_f1']:.3f}, bleached recall {r['bleached_recall']:.3f} ({r['minutes']:.1f} min)")
else:
    print("R7 skipped: fewer than two settings beat the reference, so a combination would repeat an existing run.")

sweep = pd.DataFrame([{k: v for k, v in r.items() if k not in ("hist", "stages")} for r in results])
sweep.to_csv(os.path.join(OUT, "sweep_results.csv"), index=False)
pd.concat([r["hist"].assign(run=r["name"]) for r in results]).to_csv(os.path.join(OUT, "sweep_epoch_history.csv"), index=False)
print(sweep[["name", "macro_f1", "bleached_recall", "accuracy", "best_epoch", "epochs_run", "minutes"]].round(3).to_string(index=False))

markers = ["o", "s", "^", "D", "v", "P", "X", "*"]
fig, ax = plt.subplots(figsize=(9, 4))
for i, r in enumerate(results):
    h = r["hist"]
    ax.plot(h["step"], h["macro_f1"], color=COLORS[i % len(COLORS)], marker=markers[i % len(markers)], ms=4, lw=2,
            label=f"{r['name']} (best {r['macro_f1']:.3f})")
ax.axhline(0.90, color="#888888", ls="--", lw=1); ax.text(1, 0.903, "0.90 target", fontsize=8, color="#555555")
ax.set_xlabel("Epoch (R6: stage 1 then stage 2)"); ax.set_ylabel("Validation macro F1")
ax.set_title("Run 2 tuning sweep, YOLOv8s-cls (seed 42)", fontsize=10)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=7, frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.tight_layout(); fig.savefig(os.path.join(OUT, "sweep_macro_f1.png"), dpi=150); plt.show()

## 5. Comparison model: EfficientNet-B0

EfficientNet-B0 with ImageNet weights, last layer replaced by a 2-class layer, trained on the same split with the same training loop as the scratch CNN (ImageNet normalization, AdamW lr 3e-4, cosine decay, up to 30 epochs). Section 9 of the proposal says EfficientNet-B0 becomes the final model if its validation macro F1 is higher.

In [ ]:
IMNET_MEAN, IMNET_STD = torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225])

def make_effnet():
    net = torchvision.models.efficientnet_b0(weights=EFFNET_WEIGHTS)
    net.classifier[1] = nn.Linear(net.classifier[1].in_features, 2)
    return net

eff, eff_val = None, None
try:
    t0 = time.time()
    eff, eff_hist, eff_ep = train_torch(make_effnet(), IMNET_MEAN, IMNET_STD, epochs=EPOCHS, lr=3e-4)
    eff_min = (time.time() - t0) / 60
    eff_val, _ = metrics_from(y_va, predict_torch(eff, X_va, IMNET_MEAN, IMNET_STD))
    eff_hist.to_csv(os.path.join(OUT, "effnet_b0_history.csv"), index=False)
    print(f"EfficientNet-B0 ({eff_min:.1f} min): val macro F1 {eff_val['macro_f1']:.3f}, bleached recall {eff_val['bleached_recall']:.3f}, "
          f"best epoch {eff_ep} of {EPOCHS}")
except Exception as e:
    print("EfficientNet-B0 skipped:", repr(e))

## 6. Final configuration: chosen on validation, retrained with 3 seeds

The configuration with the highest validation macro F1 (seed 42) is the final one. It is retrained with seeds 43 and 44 so the results can be reported as mean and standard deviation, as planned in the success criteria.

In [ ]:
yolo_best = sweep.loc[sweep["macro_f1"].idxmax()]
USE_EFF = eff_val is not None and eff_val["macro_f1"] > yolo_best["macro_f1"]
FINAL = "EfficientNet-B0 (ImageNet weights)" if USE_EFF else f"YOLOv8s-cls {yolo_best['name']} ({yolo_best['settings']})"
print("Final configuration (highest validation macro F1):", FINAL)

final = []   # one entry per seed: predict(files) returns P(bleached)
for s in FINAL_SEEDS:
    if USE_EFF:
        net = eff if s == SEED else train_torch(make_effnet(), IMNET_MEAN, IMNET_STD, epochs=EPOCHS, lr=3e-4, seed=s)[0]
        os.makedirs(os.path.join(BASE, "runs2"), exist_ok=True)
        path = os.path.join(BASE, "runs2", f"final_effnet_b0_s{s}.pt"); torch.save(net.state_dict(), path)
        final.append(dict(seed=s, weights=path, net=net,
                          predict=lambda files, n=net: predict_torch(n, load_tensor(files), IMNET_MEAN, IMNET_STD)))
    else:
        base_run = next(x for x in results if x["name"] == yolo_best["name"])
        r = base_run if s == SEED else yolo_run(yolo_best["name"], base_run["stages"], seed=s)
        final.append(dict(seed=s, weights=r["weights"], predict=lambda files, w=r["weights"]: predict_yolo(w, files)))

val_seeds = pd.DataFrame([dict(seed=f["seed"], **metrics_from(y_va, f["predict"](va_files))[0]) for f in final])
val_seeds.to_csv(os.path.join(OUT, "final_val_by_seed.csv"), index=False)
print(val_seeds.round(3).to_string(index=False))

## 7. Held-out test set (evaluated once)

Nothing above looked at the test set. Each seed of the final configuration is evaluated once; the scratch CNN and the majority-class check are evaluated on the same images for the baseline comparison in Objective 1.

In [ ]:
def mean_std(frame, cols=("macro_f1", "bleached_recall", "accuracy", "roc_auc")):
    return {c: (float(frame[c].mean()), float(frame[c].std(ddof=1)) if len(frame) > 1 else 0.0) for c in cols}

def plot_cm(cm, title, path):
    fig, ax = plt.subplots(figsize=(3.6, 3.2))
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=12, color="white" if v > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], CLASSES); ax.set_yticks([0, 1], CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontsize=9)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.show()

test_rows, test_pred = [], {}
for f in final:
    p = f["predict"](te_files); m, yp = metrics_from(y_te, p)
    test_rows.append(dict(seed=f["seed"], **m)); test_pred[f["seed"]] = (p, yp)
test_seeds = pd.DataFrame(test_rows); test_seeds.to_csv(os.path.join(OUT, "test_by_seed.csv"), index=False)
TEST = mean_std(test_seeds)
cnn_test, _ = metrics_from(y_te, predict_torch(cnn, X_te, MEAN, STD))
maj_test, _ = metrics_from(y_te, np.full(len(y_te), float(maj)))
print(test_seeds.round(3).to_string(index=False))
print("Mean ± std:", {k: f"{m:.3f} ± {s:.3f}" for k, (m, s) in TEST.items()})
print(f"Scratch CNN on test: macro F1 {cnn_test['macro_f1']:.3f}, bleached recall {cnn_test['bleached_recall']:.3f} | "
      f"majority class: macro F1 {maj_test['macro_f1']:.3f}")

p42, yp42 = test_pred[FINAL_SEEDS[0]]
CM_TEST = confusion_matrix(y_te, yp42, labels=[0, 1])
print(classification_report(y_te, yp42, labels=[0, 1], target_names=CLASSES, digits=3))
print("Confusion matrix (rows true, columns predicted; healthy, bleached):", CM_TEST.tolist())
plot_cm(CM_TEST, f"Held-out test, seed {FINAL_SEEDS[0]} (n={len(y_te)})", os.path.join(OUT, "test_confusion_matrix.png"))

err = pd.DataFrame({"file": [os.path.basename(p) for p in te_files], "true": [CLASSES[i] for i in y_te],
                    "pred": [CLASSES[i] for i in yp42], "p_bleached": np.round(p42, 3)})
err = err[err["true"] != err["pred"]].sort_values("p_bleached")
err.to_csv(os.path.join(OUT, "test_errors.csv"), index=False)
print(len(err), "misclassified test images saved to test_errors.csv")

## 8. External test set: BU + BHD (evaluated once)

Healthy and bleached ("unhealthy") images from the BU and BHD datasets in the Hugging Face Coral Health Classification set (dead corals and the EILAT images are left out). Each image's label and source are read from its file and folder names (`label_dataset_ID.png` on the dataset card); the counts printed below should match the card. Images that match any primary-dataset image within the near-duplicate distance are removed first, so the set is truly unseen.

In [ ]:
EXT = None
try:
    import re
    from collections import Counter
    if EXT_SRC is None:
        from huggingface_hub import hf_hub_download
        EXT_SRC = hf_hub_download("esahit/coral-health-classification", "total_dataset.zip", repo_type="dataset")
    ext_dir = EXT_SRC
    if str(EXT_SRC).lower().endswith(".zip"):
        ext_dir = os.path.join(BASE, "ext_raw"); shutil.rmtree(ext_dir, ignore_errors=True)
        with zipfile.ZipFile(EXT_SRC) as z: z.extractall(ext_dir)
        for _ in range(3):                                   # unpack any zips stored inside the zip
            inner = glob.glob(os.path.join(ext_dir, "**", "*.zip"), recursive=True)
            if not inner: break
            for zp in inner:
                with zipfile.ZipFile(zp) as z: z.extractall(os.path.splitext(zp)[0])
                os.remove(zp)
    files_all = sorted(p for p in glob.glob(os.path.join(ext_dir, "**", "*"), recursive=True) if os.path.isfile(p))
    print("Files by extension:", dict(Counter(os.path.splitext(p)[1].lower() for p in files_all)))
    print("Example names:", [os.path.relpath(p, ext_dir) for p in files_all[::max(1, len(files_all) // 10)]])

    def parse_ext(p):
        # label and source from the file name and folder names, matched as text so separators do not matter
        s = os.path.relpath(p, ext_dir).lower()
        if "unhealthy" in s or ("bleach" in s and "unbleach" not in s): lab = "bleached"
        elif "healthy" in s or "unbleach" in s: lab = "healthy"
        elif "dead" in s: lab = "dead"
        else: lab = None
        src = "BHD" if ("bhd" in s or "hbd" in s) else "EILAT" if "eilat" in s else "BU" if re.search(r"(?<![a-z])bu(?![a-z])", s) else None
        return lab, src

    img_exts = tuple(set(exts) | {".tif", ".tiff"})
    rows = [(p, *parse_ext(p)) for p in files_all if p.lower().endswith(img_exts) and "__macosx" not in p.lower()]
    ext = pd.DataFrame(rows, columns=["path", "label", "source"])
    print("All external images by source and label:")
    print(pd.crosstab(ext["source"].fillna("unknown"), ext["label"].fillna("unknown")))
    print("Dataset card: BU 124 healthy / 134 bleached; BHD 537 healthy / 374 bleached / 150 dead; EILAT 280 dead")
    ext = ext[ext["label"].isin(["healthy", "bleached"]) & ext["source"].isin(["BU", "BHD"])].reset_index(drop=True)
    ext["phash"] = ext["path"].map(phash_of)
    ext = ext.dropna(subset=["phash"])
    n_ext_raw = len(ext)
    ext = ext.drop_duplicates("phash").reset_index(drop=True)
    n_within = n_ext_raw - len(ext)
    overlap = hamming(ext["phash"], df["phash"]).min(1) <= NEAR_DUP_T     # against every primary image, all splits
    print(f"BU + BHD healthy/bleached: {n_ext_raw}; removed {n_within} repeats inside the set and {int(overlap.sum())} "
          f"matches with the Kaggle images (by source: {ext.loc[overlap, 'source'].value_counts().to_dict()}); {int((~overlap).sum())} left")
    ext = ext[~overlap].reset_index(drop=True)

    EXT_ROOT = os.path.join(BASE, "coral_external"); shutil.rmtree(EXT_ROOT, ignore_errors=True)
    for i, r in ext.iterrows():
        d = os.path.join(EXT_ROOT, "ext", r["label"]); os.makedirs(d, exist_ok=True)
        pad_square(r["path"]).save(os.path.join(d, f"{r['source']}_{i:05d}.jpg"), quality=95)
    ex_files, y_ex = split_files("ext", EXT_ROOT)
    ex_src = np.array([os.path.basename(p).split("_")[0] for p in ex_files])

    ext_rows, src_rows, ext_pred = [], [], {}
    for f in final:
        p = f["predict"](ex_files); ext_pred[f["seed"]] = p
        ext_rows.append(dict(seed=f["seed"], **metrics_from(y_ex, p)[0]))
        for s in ("BU", "BHD"):
            m = ex_src == s
            if m.sum():
                src_rows.append(dict(seed=f["seed"], source=s, n=int(m.sum()), **metrics_from(y_ex[m], p[m])[0]))
    ext_seeds = pd.DataFrame(ext_rows); ext_seeds.to_csv(os.path.join(OUT, "external_by_seed.csv"), index=False)
    src_seeds = pd.DataFrame(src_rows); src_seeds.to_csv(os.path.join(OUT, "external_by_source_and_seed.csv"), index=False)
    pe42 = ext_pred[FINAL_SEEDS[0]]
    CM_EXT = confusion_matrix(y_ex, (pe42 >= 0.5).astype(int), labels=[0, 1])
    EXT = dict(n=len(y_ex), n_bleached=int(y_ex.sum()), n_healthy=int((y_ex == 0).sum()), removed_within=n_within,
               removed_overlap=int(overlap.sum()), sources=pd.Series(ex_src).value_counts().to_dict(), by_seed=mean_std(ext_seeds),
               by_source={s: metrics_from(y_ex[ex_src == s], pe42[ex_src == s])[0] for s in ("BU", "BHD") if (ex_src == s).sum()},
               by_source_ms={s: dict(n=int(g["n"].iloc[0]), **mean_std(g)) for s, g in src_seeds.groupby("source")}, cm=CM_EXT.tolist())
    print(ext_seeds.round(3).to_string(index=False))
    print("All (mean ± std):", {k: f"{m:.3f} ± {s:.3f}" for k, (m, s) in EXT["by_seed"].items()})
    for s, ms in EXT["by_source_ms"].items():
        print(f"{s} (n={ms['n']}):", {k: f"{v[0]:.3f} ± {v[1]:.3f}" for k, v in ms.items() if k != "n"})
    print("Seed", FINAL_SEEDS[0], "confusion matrix (rows true, columns predicted; healthy, bleached):", CM_EXT.tolist())
    plot_cm(CM_EXT, f"External BU + BHD, seed {FINAL_SEEDS[0]} (n={len(y_ex)})", os.path.join(OUT, "external_confusion_matrix.png"))
except Exception as e:
    print("External test skipped:", repr(e))

## 9. Model size and CPU latency

Same protocol as the success criteria: mean time per image over 100 test images, batch size 1, after 10 warm-up runs, on the CPU. This runs on the Colab CPU; repeat it on a team laptop for the final report.

In [ ]:
def cpu_latency_ms(predict_one, files, n=100, warm=10):
    for p in files[:warm]: predict_one(p)
    t0 = time.perf_counter()
    for p in files[:n]: predict_one(p)
    return (time.perf_counter() - t0) / len(files[:n]) * 1000

def torch_predict_one(net, mean, std):
    def f(p):
        x = torch.from_numpy(np.array(Image.open(p).convert("RGB").resize((IMGSZ, IMGSZ), Image.BILINEAR))).permute(2, 0, 1).float()[None] / 255
        with torch.no_grad():
            return net((x - mean[:, None, None]) / std[:, None, None]).softmax(1)
    return f

f42 = final[0]
FINAL_PATH = os.path.join(OUT, "final_model_s42" + (".pt" if not USE_EFF else "_effnet_b0.pt"))
shutil.copy(f42["weights"], FINAL_PATH)
SIZE_MB = os.path.getsize(FINAL_PATH) / 1e6
# model card: architecture, class order and preprocessing saved with the weights (read by demo.py)
CARD = dict(arch="efficientnet_b0" if USE_EFF else "yolov8s-cls", configuration=FINAL, seed=FINAL_SEEDS[0], imgsz=IMGSZ,
            preprocessing="pad to square with black borders, resize to imgsz (bilinear), RGB scaled to [0, 1], then (x - mean) / std",
            classes=CLASSES if USE_EFF else [YOLO(FINAL_PATH).names[i] for i in range(2)],
            mean=IMNET_MEAN.tolist() if USE_EFF else [0.0, 0.0, 0.0], std=IMNET_STD.tolist() if USE_EFF else [1.0, 1.0, 1.0],
            val_macro_f1=float(val_seeds.iloc[0]["macro_f1"]), val_bleached_recall=float(val_seeds.iloc[0]["bleached_recall"]),
            ultralytics=__import__("ultralytics").__version__, torch=torch.__version__)
json.dump(CARD, open(os.path.splitext(FINAL_PATH)[0] + ".json", "w"), indent=2)
if USE_EFF:
    LAT_MS = cpu_latency_ms(torch_predict_one(copy.deepcopy(f42["net"]).cpu().eval(), IMNET_MEAN, IMNET_STD), te_files)
else:
    mc = YOLO(FINAL_PATH)
    LAT_MS = cpu_latency_ms(lambda p: mc.predict(p, imgsz=IMGSZ, device="cpu", verbose=False), te_files)
CNN_SIZE_MB = os.path.getsize(CNN_PATH) / 1e6
CNN_LAT_MS = cpu_latency_ms(torch_predict_one(copy.deepcopy(cnn).cpu().eval(), MEAN, STD), te_files)
print(f"Final model: {SIZE_MB:.1f} MB, {LAT_MS:.0f} ms/image on CPU ({os.cpu_count()} cores) | "
      f"scratch CNN: {CNN_SIZE_MB:.1f} MB, {CNN_LAT_MS:.0f} ms/image")

## 10. Error analysis: Grad-CAM heatmaps

Grad-CAM for the predicted class on held-out test images (seed 42 model): correct bleached, correct healthy, and misclassified examples. Red marks the regions that drove the prediction, which shows whether the model looks at the coral or at the water, divers, or fish. The same function can produce the heatmap in the demo.

In [ ]:
def gradcam(net, layer, x):
    """Grad-CAM for the predicted class: weight the layer's activation maps by their mean gradients."""
    store = {}
    hook = layer.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    out = net(x); hook.remove()
    logits = out[1] if isinstance(out, (tuple, list)) else out
    cls = int(logits.argmax(1)); a = store["a"]; a.retain_grad()
    net.zero_grad(); logits[0, cls].backward()
    cam = F.relu((a.grad.mean((2, 3), keepdim=True) * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0].detach().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), cls, float(logits.softmax(1)[0, cls].detach())

if USE_EFF:
    cam_net = copy.deepcopy(f42["net"]).cpu().eval(); cam_layer = cam_net.features[-1]
    cam_norm = lambda x: (x - IMNET_MEAN[:, None, None]) / IMNET_STD[:, None, None]
    cam_name = lambda c: CLASSES[c]
else:
    cam_net = YOLO(FINAL_PATH).model.float().eval(); cam_layer = cam_net.model[-1].conv
    cam_norm = lambda x: x                                # Ultralytics classification input is RGB in [0, 1]
    cam_name = lambda c: cam_net.names[c]
for prm in cam_net.parameters(): prm.requires_grad_(True)

def heatmap(path):
    img = np.array(Image.open(path).convert("RGB").resize((IMGSZ, IMGSZ), Image.BILINEAR))
    cam, c, prob = gradcam(cam_net, cam_layer, cam_norm(torch.from_numpy(img).permute(2, 0, 1).float()[None] / 255))
    return img, cam, cam_name(c), prob

pick = {"correct bleached": [i for i in range(len(y_te)) if y_te[i] == 1 and yp42[i] == 1][:4],
        "correct healthy": [i for i in range(len(y_te)) if y_te[i] == 0 and yp42[i] == 0][:4],
        "misclassified": [i for i in range(len(y_te)) if y_te[i] != yp42[i]][:4]}
rows = [(k, v) for k, v in pick.items() if v]
fig, axes = plt.subplots(len(rows), 4, figsize=(11, 2.9 * len(rows)), squeeze=False)
for ax in axes.ravel(): ax.axis("off")
red = np.array([0.89, 0.29, 0.28])
for r_i, (kind, idx) in enumerate(rows):
    for c_i, i in enumerate(idx):
        img, cam, pred, prob = heatmap(te_files[i])
        overlay = np.dstack([np.broadcast_to(red, cam.shape + (3,)), cam * 0.65])
        ax = axes[r_i, c_i]; ax.imshow(img); ax.imshow(overlay)
        ax.set_title(f"{kind}\ntrue {CLASSES[y_te[i]]}, pred {pred} ({prob:.2f})", fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "gradcam_test.png"), dpi=150); plt.show()

## 11. Summary for the progress log

Copy the text printed below into the chat or the progress report. The zip in the Files panel has every table and figure.

In [ ]:
def fmt(ms): return f"{ms[0]:.3f} ± {ms[1]:.3f}"
VAL = mean_std(val_seeds)
target = lambda ok: "met" if ok else "NOT met"
lines = [
    f"RUN 2 SUMMARY ({time.strftime('%b. %d, %Y')})",
    f"Data: {N_RAW} images; Run 1 split reused: {SAME_SPLIT}. Conflicting-label duplicate groups: {len(conflict_hashes)} "
    f"({len(conf)} files); copies removed from the split: {removed_conflict['split'].value_counts().to_dict()}.",
    f"Near-duplicates (pHash distance <= {NEAR_DUP_T}): {N_NEAR_PAIRS} pairs; {N_CROSS} same-label pairs across splits; "
    f"{N_NEAR_CONFLICT} pairs with different labels; removed {NEAR_REMOVED}.",
    "Final split (n, bleached, healthy): " + "; ".join(f"{s} {c['n']} ({c['bleached']}, {c['healthy']})" for s, c in COUNTS.items()),
    f"Majority-class check: val macro F1 {maj_val['macro_f1']:.3f}, test {maj_test['macro_f1']:.3f}.",
    f"Scratch CNN ({cnn_params / 1e6:.2f}M params, best epoch {cnn_ep} of {CNN_EPOCHS}): val macro F1 {cnn_val['macro_f1']:.3f}, "
    f"bleached recall {cnn_val['bleached_recall']:.3f}; test macro F1 {cnn_test['macro_f1']:.3f}, bleached recall {cnn_test['bleached_recall']:.3f}; "
    f"{CNN_SIZE_MB:.1f} MB, {CNN_LAT_MS:.0f} ms/image CPU.",
    "Sweep (YOLOv8s-cls, seed 42, validation at the best-macro-F1 epoch):",
]
for r in results:
    lines.append(f"  {r['name']}: macro F1 {r['macro_f1']:.3f}, bleached recall {r['bleached_recall']:.3f}, acc {r['accuracy']:.3f}, "
                 f"epoch {r['best_epoch']} (stage {r['best_stage']}), {r['epochs_run']} epochs run, {r['minutes']:.1f} min [{r['settings']}]")
lines.append("EfficientNet-B0: " + (f"val macro F1 {eff_val['macro_f1']:.3f}, bleached recall {eff_val['bleached_recall']:.3f}, best epoch {eff_ep}"
                                    if eff_val is not None else "skipped"))
lines += [
    f"Final configuration: {FINAL}.",
    f"Validation over seeds {FINAL_SEEDS}: macro F1 {fmt(VAL['macro_f1'])}, bleached recall {fmt(VAL['bleached_recall'])}.",
    f"Held-out test (n={len(y_te)}, evaluated once): macro F1 {fmt(TEST['macro_f1'])}, bleached recall {fmt(TEST['bleached_recall'])}, "
    f"accuracy {fmt(TEST['accuracy'])}, ROC AUC {fmt(TEST['roc_auc'])}.",
    f"  By seed: " + "; ".join(f"{int(r.seed)}: F1 {r.macro_f1:.3f}, recall {r.bleached_recall:.3f}" for r in test_seeds.itertuples()),
    f"  Seed {FINAL_SEEDS[0]} confusion matrix (rows true, cols predicted; healthy, bleached): {CM_TEST.tolist()}; {len(err)} errors.",
    f"  Gain over the scratch CNN on test: {100 * (TEST['macro_f1'][0] - cnn_test['macro_f1']):+.1f} macro F1 points.",
]
if EXT is not None:
    lines += [f"External BU + BHD (n={EXT['n']}: {EXT['n_bleached']} bleached, {EXT['n_healthy']} healthy; removed {EXT['removed_overlap']} "
              f"matches with the primary set and {EXT['removed_within']} repeats; evaluated once): macro F1 {fmt(EXT['by_seed']['macro_f1'])}, "
              f"bleached recall {fmt(EXT['by_seed']['bleached_recall'])}, accuracy {fmt(EXT['by_seed']['accuracy'])}.",
              "  By source (mean ± std over seeds): " + "; ".join(f"{s} (n={ms['n']}) F1 {fmt(ms['macro_f1'])}, recall {fmt(ms['bleached_recall'])}"
                                                               for s, ms in EXT["by_source_ms"].items())
              + f"; seed {FINAL_SEEDS[0]} confusion matrix {EXT['cm']}."]
else:
    lines.append("External BU + BHD: skipped (see the error printed in section 8).")
lines += [
    f"Size {SIZE_MB:.1f} MB; CPU latency {LAT_MS:.0f} ms/image (Colab CPU, batch 1, 100 test images after 10 warm-ups).",
    f"Targets on test: macro F1 >= 0.90 {target(TEST['macro_f1'][0] >= 0.90)}; bleached recall >= 0.90 {target(TEST['bleached_recall'][0] >= 0.90)}; "
    f"size < 50 MB {target(SIZE_MB < 50)}; latency < 200 ms {target(LAT_MS < 200)}; "
    f"+10 points over the baseline {target(TEST['macro_f1'][0] - cnn_test['macro_f1'] >= 0.10)}.",
    f"Environment: GPU {GPU_NAME}, ultralytics {__import__('ultralytics').__version__}, torch {torch.__version__}. "
    f"Total time {(time.time() - T_START) / 60:.1f} min.",
]
summary = "\n".join(lines)
print(summary)
open(os.path.join(OUT, "run2_summary.txt"), "w").write(summary)
json.dump(dict(final=FINAL, val=VAL, test=TEST, cnn_test=cnn_test, maj_test=maj_test, ext=EXT, size_mb=SIZE_MB, latency_ms=LAT_MS,
               sweep=sweep.drop(columns=["weights"]).to_dict("records"), counts=COUNTS),
          open(os.path.join(OUT, "summary.json"), "w"), indent=2, default=str)

ZIP = shutil.make_archive(os.path.join(BASE, "run2_results"), "zip", OUT)
print("\nSaved", ZIP)
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    pass